In [32]:
import pygad
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
import json
from fitness import *

In [33]:
with open("datasets/medium_dataset_2.json", "r") as file:
    data = json.load(file)

# Step 2: Extract information
dataset_id = data["dataset_id"]
num_papers = data["num_papers"]
num_reviewers = data["num_reviewers"]
reviewer_capacity = data["reviewer_capacity"]
min_reviews_per_paper = data["min_reviews_per_paper"]
max_reviews_per_paper = data["max_reviews_per_paper"]
preferences = np.array(data["preferences"])
friendships = np.array(data["friendships"])
authorship = np.array(data["authorship"])

## Analytic solver

In [34]:
from pulp import LpMaximize, LpProblem, LpVariable, lpSum, value

# Define the ILP problem
problem = LpProblem("PaperAssignment", LpMaximize)

# Decision Variables: x_ij (binary)
x = [[LpVariable(f"x_{i}_{j}", cat="Binary") for j in range(num_papers)] for i in range(num_reviewers)]

# Slack Variables for soft constraints
slack_min_reviews = [LpVariable(f"slack_min_reviews_{j}", lowBound=0) for j in range(num_papers)]
slack_max_reviews = [LpVariable(f"slack_max_reviews_{j}", lowBound=0) for j in range(num_papers)]
slack_capacity = [LpVariable(f"slack_capacity_{i}", lowBound=0) for i in range(num_reviewers)]
slack_authorship = [[LpVariable(f"slack_authorship_{i}_{j}", lowBound=0) for j in range(num_papers)] for i in range(num_reviewers)]
slack_friendship = [[LpVariable(f"slack_friendship_{i}_{j}", lowBound=0) for j in range(num_papers)] for i in range(num_reviewers)]

# Objective Function: Maximize preferences and minimize penalties for violations
penalty_weight = 10  # Weight for penalties
problem += (
    lpSum(preferences[i][j] * x[i][j] for i in range(num_reviewers) for j in range(num_papers))
    - penalty_weight * (lpSum(slack_min_reviews) + lpSum(slack_max_reviews) + lpSum(slack_capacity)
                        + lpSum(slack_authorship[i][j] for i in range(num_reviewers) for j in range(num_papers))
                        + lpSum(slack_friendship[i][j] for i in range(num_reviewers) for j in range(num_papers)))
)

# Constraints
# 1. Minimum and Maximum Reviews per Paper (softened)
for j in range(num_papers):
    num_reviews = lpSum(x[i][j] for i in range(num_reviewers))
    problem += num_reviews + slack_min_reviews[j] >= min_reviews_per_paper  # Allow slack for minimum
    problem += num_reviews - slack_max_reviews[j] <= max_reviews_per_paper  # Allow slack for maximum

# 2. Reviewer Capacity (softened)
for i in range(num_reviewers):
    num_papers_reviewed = lpSum(x[i][j] for j in range(num_papers))
    problem += num_papers_reviewed - slack_capacity[i] <= reviewer_capacity  # Allow slack

# 3. Authorship Constraint (softened)
for i in range(num_reviewers):
    for j in range(num_papers):
        if authorship[i][j] == 1:
            problem += x[i][j] - slack_authorship[i][j] <= 0  # Allow slack for authorship violations

# 4. Friendship Constraint (softened)
for r1 in range(num_reviewers):
    for r2 in range(num_reviewers):
        if friendships[r1][r2] == 1:
            for j in range(num_papers):
                problem += x[r1][j] + x[r2][j] - 1 - slack_friendship[r1][j] <= 0  # Allow slack for friendship violations

# Solve the ILP
problem.solve()

# Extract the solution
assignment = np.zeros((num_reviewers, num_papers), dtype=int)
for i in range(num_reviewers):
    for j in range(num_papers):
        assignment[i][j] = int(value(x[i][j]))  # Convert to integer (0 or 1)

# Print the result
print("Assignment Matrix:")
print(assignment)

# Check for violations and penalties
penalties = {
    "min_reviews_violation": sum(value(slack_min_reviews[j]) for j in range(num_papers)),
    "max_reviews_violation": sum(value(slack_max_reviews[j]) for j in range(num_papers)),
    "capacity_violation": sum(value(slack_capacity[i]) for i in range(num_reviewers)),
    "authorship_violation": sum(value(slack_authorship[i][j]) for i in range(num_reviewers) for j in range(num_papers)),
    "friendship_violation": sum(value(slack_friendship[i][j]) for i in range(num_reviewers) for j in range(num_papers)),
}

print("\nPenalties:")
for key, penalty in penalties.items():
    print(f"{key}: {penalty:.2f}")


Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/matjaz/.conda/envs/IS/lib/python3.13/site-packages/pulp/solverdir/cbc/linux/64/cbc /tmp/f389f8751539423baf3dd51c8d3188bd-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /tmp/f389f8751539423baf3dd51c8d3188bd-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 219 COLUMNS
At line 1574 RHS
At line 1789 BOUNDS
At line 1886 ENDATA
Problem MODEL has 214 rows, 320 columns and 852 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 130.833 - 0.00 seconds
Cgl0004I processed model has 214 rows, 238 columns (96 integer (96 of which binary)) and 852 elements
Cutoff increment increased from 1e-05 to 0.9999
Cbc0038I Initial state - 22 integers unsatisfied sum - 7.33333
Cbc0038I Pass   1: suminf.    0.00000 (0) obj. -67 iterations 72
Cbc0038I Solution found of -67
Cbc0038I Relaxing co

In [35]:
print(eval_ff(assignment, preferences, num_reviewers, num_papers, max_reviews_per_paper, min_reviews_per_paper, reviewer_capacity, authorship, friendships))

130
